# GaussLaw — projection of a non-solenoidal magnetic field

`MagneticStandalone` · opt-in Gauss-law correction.

This form starts from a compatible but non-solenoidal vector field, enables `correction_divergence`, and monitors the reported `divergence_magnetique` through the initial projection and subsequent time steps. It runs the same template on the coarse/standard/fine mesh ladder and on two MPI ranks.

In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Run the mesh-convergence ladder

All scales use the one dataset template `src/gauss_law.data`. The `PARALLEL OK` markers in the dataset are required for the 2-rank run.

In [ ]:
import os
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY
NB_PROCS = 2
run.reset()
meshes = {
    "coarse": {"nx": 17, "ny": 17},
    "standard": {"nx": 33, "ny": 33},
    "fine": {"nx": 65, "ny": 65},
}
cases = {}
for level, mesh in meshes.items():
    cases[level] = run.addCaseFromTemplate("gauss_law.data", level, dic=mesh,
                                           nbProcs=NB_PROCS, excluNR=True)
    cases[level].partition()
run.printCases()
run.runCases()


## Gauss-law checks

The field is initially `b=(cos(2*pi*x), 0)`: its Green–Gauss divergence is nonzero locally, while the net boundary flux is zero. The correction is applied during initialization and after each time-step update. The three probes are strictly interior to avoid the boundary-cell gradient reconstruction.

In [ ]:
def read_probe(level, tag):
    case_name = cases[level].dataFileName_
    candidates = (f"{BUILD}/{level}/{case_name}_{tag}.son",
                  f"{BUILD}/{case_name}_{tag}.son")
    path = next((candidate for candidate in candidates if os.path.exists(candidate)), None)
    assert path is not None, f"missing probe file: {candidates}"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"empty probe file: {path}"
    t, values = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(values, dtype=float)

TAGS = ("SONDE_DIV_1", "SONDE_DIV_2", "SONDE_DIV_3")
for level in ("coarse", "standard", "fine"):
    worst = 0.0
    for tag in TAGS:
        t, values = read_probe(level, tag)
        worst = max(worst, float(np.max(np.abs(values))))
    print(f"{level:>8}: max |divergence_magnetique| = {worst:.4e}")
    assert worst < 1.e-8, f"Gauss-law correction failed on {level}: {worst:.3e}"
print("OK: interior magnetic divergence remains at solver tolerance across the mesh scan.")


## Scope

This projection enforces the module’s Green–Gauss divergence measure. It does not claim that the separate MPFA diffusive-flux divergence is identically zero. The unit tests additionally verify `L phi = D(G phi)`, one-pass removal of a compatible synthetic divergence, idempotence, and rejection of nonzero net boundary flux.